### Chapter 7: Search In-Depth
### Topic 6: MMR — Maximal Marginal Relevance

### Why Do We Need MMR?

BM25, Dense Retrieval, and Hybrid Search rank documents **independently**.

They ask:

> **"How relevant is this document to the query?"**

They do **not** ask:

> **"Does another selected document already contain the same information?"**

This can lead to duplicate results.

---

### Example Without MMR

User asks:

```text
What is the penalty for premature FD withdrawal?
```

Retriever returns:

```text
1. FAQ
Premature FD withdrawal incurs a 1% penalty.

2. Policy
Premature FD withdrawal incurs a 1% penalty.

3. User Guide
Premature FD withdrawal incurs a 1% penalty.
```

All three documents are relevant.

But they all say the **same thing**.

The user learns only one fact.

---

### What Is MMR?

**Definition:** MMR (Maximal Marginal Relevance) is a **re-ranking algorithm** that selects documents which are both **relevant** and **different** from documents already selected.

### Logic

MMR asks two questions:

1. **Is this document relevant to the query?**
2. **Does it provide new information, or is it repeating what I already selected?**

A document is selected only if it performs well on both.

---

### Example With MMR

Initial retrieval:

```text
1. FAQ
1% penalty

2. Policy
1% penalty

3. Exception
Senior citizens are exempt from the penalty.
```

Without MMR:

```text
FAQ
Policy
Procedure
```

All repeat the same information.

With MMR:

```text
FAQ
Exception
Policy
```

The second document is chosen because it introduces **new information**, even though it may have a slightly lower relevance score.

---

### Why Is It Called "Marginal" Relevance?

"Marginal" means:

> **The additional value a document provides after considering what has already been selected.**

For example:

The first document says:

```text
Penalty = 1%
```

The second document also says:

```text
Penalty = 1%
```

It adds almost **no new information**.

Its marginal value is low.

Another document says:

```text
Senior citizens are exempt from the penalty.
```

This adds completely new information.

Its marginal value is high.

So MMR prefers it.

---

### Key Idea

The biggest confusion is that MMR is not another retrieval algorithm like BM25 or Dense Retrieval. MMR works after retrieval.

Think of it like this:
- BM25 / Dense Retrieval: "Which documents are relevant?"
- MMR: "Among these relevant documents, which combination gives the most useful information?"

- **Retriever:** Finds the most relevant documents.
- **MMR:** Removes redundancy by preferring documents that add new information.

MMR does **not** retrieve new documents.

It simply **re-orders the documents that have already been retrieved** to produce a more diverse and informative result set.

---


### Internal Working — Step by Step

Suppose the retriever has already returned these documents.

```text
1. FAQ
Penalty = 1%

2. Policy
Penalty = 1%

3. Procedure
Penalty = 1%

4. Exception
Senior citizens are exempt.

5. Charges
Penalty is waived for staff accounts.
```

MMR now decides which three documents should finally be shown.

---

### Step 1: Select the First Document

Nothing has been selected yet.

So MMR simply chooses the **most relevant** document.

```text
Selected

FAQ
Penalty = 1%
```

---

### Step 2: Choose the Second Document

Now MMR evaluates the remaining documents.

**Policy**

```text
Penalty = 1%
```

- Relevant to the query: Yes
- Adds new information: No

---

**Procedure**

```text
Penalty = 1%
```

- Relevant to the query: Yes
- Adds new information: No

---

**Exception**

```text
Senior citizens are exempt.
```

- Relevant to the query: Yes
- Adds new information: Yes

MMR selects:

```text
Exception
```

because it contributes new information instead of repeating the first document.

The selected list becomes:

```text
FAQ
Exception
```

---

### Step 3: Choose the Third Document

MMR evaluates the remaining documents again.

**Policy**

```text
Penalty = 1%
```

Still repeats the FAQ.

---

**Charges**

```text
Penalty is waived for staff accounts.
```

- Relevant to the query: Yes
- Adds new information: Yes

MMR selects:

```text
Charges
```

---

### Final Result

**Without MMR**

```text
FAQ
Policy
Procedure
```

All three documents repeat the same information.

---

**With MMR**

```text
FAQ
Exception
Charges
```

Each document contributes different information, making the final result more useful.

---

### Where Does the Formula Fit?

The formula simply assigns a score to every remaining document using two factors:

- **Relevance:** How well the document matches the user's query.
- **Redundancy:** How similar the document is to documents already selected.

If a document is highly relevant and different from the selected documents, it receives a high score.

If a document is highly relevant but almost identical to an already selected document, its score is reduced.

---

### What Does Lambda (λ) Control?

Lambda controls the balance between **relevance** and **diversity**.

- **λ = 1**
  - Only relevance matters.
  - MMR behaves like normal Top-K retrieval.

- **λ = 0**
  - Only diversity matters.
  - MMR keeps selecting different documents even if they are less relevant.

- **Typical values (0.5–0.8)**
  - Relevance remains the primary factor.
  - Diversity helps remove duplicate or highly similar documents.

---

### One-Line Summary

The retriever finds the most relevant documents.

MMR then re-orders those retrieved documents so that the final result contains relevant documents that also provide different information instead of repeating the same content.

---

### Real-World Considerations

**Computational Cost**

MMR works only on the documents already retrieved by the retriever.

For example:

- Total documents in Qdrant = **1,000,000**
- Retriever returns = **20 documents**
- MMR selects the best **5 documents**

MMR compares only these **20 retrieved documents**, not all 1 million documents.

Therefore, MMR is usually very fast.

**Production Perspective:** Never run MMR on the entire corpus. First retrieve a small candidate pool (e.g., Top 20–100 documents), then apply MMR. This keeps latency low while still improving diversity.

---

**Lambda (λ) Tuning**

Lambda controls the balance between **relevance** and **diversity**.

- Higher λ
  - Gives more importance to relevance.
  - Results may contain similar documents.

- Lower λ
  - Gives more importance to diversity.
  - Results contain more unique information but may be slightly less relevant.

There is no single best value. It is usually chosen by testing with real user queries.

**Production Perspective:** Teams evaluate different lambda values on representative queries and choose the one that provides the best balance between relevance and diversity for their specific application.

---

**Debugging**

Suppose the user expects this result:

```text
Senior citizens are exempt from the penalty.
```

but MMR does not return it.

Possible reasons:

- The retriever never retrieved that document.
- MMR considered it too similar to an already selected document.
- Lambda gives too much importance to diversity or relevance.

The first step is always to check whether the document was present in the retrieved candidate list.

**Production Perspective:** Always debug retrieval first, then MMR. If the document is missing from the candidate pool, MMR cannot select it because it only re-ranks retrieved documents.

---

**Monitoring**

Monitor whether MMR is actually improving the results.

For example:

Without MMR

```text
FAQ
Policy
Procedure
```

All three repeat the same information.

With MMR

```text
FAQ
Exception
Charges
```

Each document contributes different information.

If MMR continues returning many duplicate documents, the lambda value may need adjustment or the retrieved candidate pool may itself contain mostly duplicates.

**Production Perspective:** Monitor how often duplicate documents appear in the final results. A sudden increase may indicate duplicate document ingestion, poor retrieval quality, or an unsuitable lambda value.

---

**Latency**

MMR does not call another AI model.

It simply compares the already retrieved document embeddings.

Therefore, the additional processing time is usually very small.

**Production Perspective:** Since MMR reuses existing embeddings, it adds only a small amount of CPU computation and almost no additional inference cost.

---

**Deployment**

MMR is added after retrieval.

The pipeline becomes:

```text
User Query

Retriever

Top 20 Documents

MMR

Final Top 5 Documents

LLM
```

No changes are required to the Vector Database, embedding model, or LLM. MMR is simply an additional post-processing step before sending documents to the LLM.

**Production Perspective:** MMR is easy to introduce into an existing RAG pipeline because it requires no model retraining, no database changes, and no additional infrastructure. It is simply another processing step between retrieval and generation.

---

### Design Decisions, Trade-offs, and Real-Time Dilemmas

- **MMR vs simply deduplicating near-identical chunks:** a simpler alternative is to just detect and remove chunks that are above some similarity threshold to already-selected chunks — a binary duplicate-removal approach rather than MMR's continuous relevance-diversity trade-off. MMR is strictly more expressive, since it can choose a moderately redundant-but-highly-relevant document over a completely-novel-but-barely-relevant one, which a simple binary threshold cannot represent. For a corpus with mostly distinct documents and only occasional near-duplicates, simple threshold-based deduplication may capture most of the benefit with far less tuning complexity — worth evaluating both before committing to the more complex option.
- **Where in the pipeline MMR should run relative to reranking (Topic 7):** MMR re-orders for diversity, while a more accurate relevance-scoring reranker re-orders for relevance accuracy — these solve different problems, and the right order matters. A common, correct pattern: retrieve candidates first, then rerank them for more accurate relevance scores, then apply MMR on that now-more-accurately-scored pool to select a final diverse set. Running MMR before a more accurate reranking step means MMR's relevance signal is based on the less accurate initial retrieval score rather than the more accurate reranked score — generally the wrong order.
- **Fixed result-set size vs adaptive size:** a fixed-size final result set is simple, but MMR naturally supports stopping early if all remaining candidates have very low marginal value (relevance minus redundancy penalty falling below some threshold), avoiding forcing in a genuinely low-value final document just to hit a fixed count. This becomes worth considering once there's a proper way to budget how much context downstream consumption can actually use — a genuinely low-value final chunk consumes that budget without adding proportional value.


### Alternatives and When to Use Each

- **MMR (this topic's approach):** best for corpora with real content redundancy across multiple source documents — for example, the same fact restated across an FAQ, a policy document, and a procedure document. Use when relevance-only ranking is *measurably* returning near-duplicate results, verified by actually inspecting real query outputs, not just assumed.
- **Simple similarity-threshold deduplication:** best for corpora where redundancy is rare and mostly exact-or-near-exact, especially if duplicate detection is already partially handled earlier in the pipeline (like at ingestion time). Use when MMR's continuous relevance-diversity trade-off is more complexity than the actual redundancy pattern in the corpus actually warrants.
- **Clustering-based diversity (cluster candidates, then pick one representative per cluster):** best for much larger candidate pools where MMR's cost becomes a real concern, or where the corpus has clearly separable topic clusters. Use when guaranteed coverage across distinct clusters matters more than MMR's greedy, order-dependent selection process.
- **No diversity handling at all (relevance-only ranking):** best for corpora with genuinely low redundancy, or use cases where the single most relevant answer is what actually matters and secondary or complementary information has low value. Use when a qualitative review of actual retrieval outputs shows redundancy isn't really a problem in practice — don't add MMR's tuning burden without evidence that it's actually needed.


### Common Mistakes and Production Failures

- Applying MMR before actually checking whether redundancy is a real, measured problem — MMR adds a tunable parameter that needs justification from real evidence, not blind adoption because it sounds like a good idea.
- Using the wrong similarity function for the redundancy check — this must be a genuine document-to-document semantic comparison using dense embeddings, not accidentally reusing a query-relevance score or a keyword-based score that was never designed for this kind of comparison.
- Setting the diversity weight too aggressively without realizing it can hurt precision — at an extreme diversity setting, MMR essentially prioritizes being different over being relevant, which can pull in genuinely off-topic candidates purely because they happen to be dissimilar to what's already selected. Always sanity-check outputs qualitatively when tuning toward more diversity.
- Running MMR before a more accurate relevance-reranking step instead of after — this means MMR's relevance judgments are based on a less accurate initial score rather than the more refined one that a proper reranking step would provide.
- Applying MMR over an enormous, unfiltered candidate pool instead of first narrowing down with retrieval — this defeats the performance benefit of MMR being a cheap, local re-ranking step and turns it into a real computational cost.


### Lead-Level Interview Questions

**Basic**

- Q: What specific problem does MMR solve that plain relevance-based ranking cannot?  
  A: Plain relevance ranking can return several near-duplicate top results that all say essentially the same thing, because each document is scored independently of what else is in the result set. MMR explicitly penalizes redundancy with already-selected documents, allowing genuinely distinct but slightly-less-relevant information to also make it into the final result set.

- Q: What does the lambda parameter control in MMR?  
  A: It balances relevance against diversity. A value of 1 makes MMR behave like plain relevance ranking with no diversity consideration; a value of 0 makes it prioritize being different from what's already selected almost regardless of relevance; typical practical values sit in between, treating relevance as the primary driver with diversity as a secondary correction.

**Intermediate**

- Q: Why must MMR's redundancy check use dense embeddings rather than the retrieval method's original relevance score?  
  A: The redundancy check is a document-to-document semantic comparison — how similar is this candidate to something already selected — which is a fundamentally different comparison than query-to-document relevance. A keyword-based relevance score isn't designed for comparing two documents to each other; dense embeddings, which represent meaning as vectors, are the right tool for this kind of similarity check.

- Q: Why does MMR always pick the single most relevant document first, regardless of the lambda value?  
  A: Because the redundancy term depends on similarity to already-selected documents, and on the very first pick, nothing has been selected yet — so that term is zero for every candidate, and the formula reduces to pure relevance ranking for that first pick only.

**Advanced**

- Q: A colleague suggests applying MMR before a more accurate relevance-reranking step instead of after. What's wrong with this ordering?  
  A: MMR's relevance term relies on whatever relevance score is available at the time it runs. If MMR runs before the more accurate reranking step, it's making its diversity trade-off decisions based on the less accurate initial retrieval score rather than the more refined one. The correct order is to rerank for accurate relevance first, then apply MMR on that more accurately-scored pool to make the final diversity-aware selection.

- Q: How would you decide between full MMR and simple similarity-threshold deduplication for a given corpus?  
  A: Inspect actual retrieval outputs for real queries. If the corpus mostly returns clearly distinct documents with only rare exact-or-near-exact duplicates, simple threshold-based deduplication likely captures most of the benefit with far less tuning complexity. If the corpus has genuine structural redundancy — the same information restated multiple times across different documents with real but partial relevance differences — MMR's continuous trade-off is worth the added complexity, since simple deduplication can't represent "moderately redundant but highly relevant" versus "completely novel but barely relevant" the way MMR's formula can.

**Scenario-based**

- Q: Monitoring shows the average pairwise similarity within your top-5 results has been rising over the past month. Walk through your diagnosis. 
  A: First check whether new content was recently added to the knowledge base that duplicates existing information — this would organically increase redundancy in the candidate pool being fed to MMR. If the candidate pool composition hasn't meaningfully changed, consider whether the lambda weight needs adjusting toward more diversity. Also spot-check a sample of actual queries and their returned results to confirm the metric reflects a real user-facing problem, not just noise in a handful of edge-case queries.

**Follow-up questions to expect:**

- "How would you measure whether MMR is actually improving downstream answer quality, not just diversifying results for its own sake?"
- "What happens to MMR's behavior if the candidate pool itself is too small or too homogeneous?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Greedy, order-dependent selection:** MMR builds its result set one document at a time, and each choice depends on everything already chosen — this means the algorithm makes locally-good decisions at each step, but doesn't guarantee the globally best possible combination of documents. This greedy nature is a known, accepted trade-off for the speed and simplicity it provides.
- **The relevance-diversity trade-off is a general pattern, not unique to retrieval:** the same underlying tension — optimizing for individual quality versus optimizing for a good combination as a whole — shows up in many other contexts, like recommendation systems trying to avoid showing a user five nearly-identical items in a row.
- **MMR's dependence on the quality of the initial candidate pool:** MMR can only select from what it's given — if the initial retrieval pass already failed to surface a genuinely relevant and distinct document, no amount of diversity-aware re-ranking can recover it. This reinforces why MMR is a re-ranking step on top of good retrieval, not a substitute for it.

### 10. Quick Revision Summary (for last-minute interview prep)

> MMR fixes a blind spot that every purely relevance-based retrieval method shares: individually scoring documents against a query, with no awareness of redundancy across the result set, can produce a top-k list full of near-duplicates while genuinely useful complementary information gets excluded. MMR builds its result set one document at a time, at each step picking whichever remaining candidate offers the best combined score of relevance to the query minus similarity to what's already been selected, controlled by a tunable weight (lambda) that balances the two. The first pick is always purely the most relevant document, since nothing has been selected yet to be redundant with. This is a cheap, local re-ranking step applied after retrieval — it needs a genuine document-to-document similarity check (dense embeddings) for its redundancy term, should run after (not before) any more accurate relevance-reranking step, and should only be adopted once redundancy is confirmed as a real, measured problem in actual retrieval outputs — not applied by default just because it sounds beneficial.


### Module 1: Setup — A Candidate Pool With Real Redundancy

Build a candidate pool where some documents are near-duplicates of each other (restating the same fact), using offline TF-IDF+SVD dense vectors for both query-relevance and document-to-document similarity.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- candidate pool with real redundancy
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# 3 of these chunks restate the SAME penalty fact in different words
# (FAQ style, policy style, SOP style) -- deliberate redundancy.
# The other 2 are genuinely different, secondary information.
CANDIDATE_POOL = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",          # 0: FAQ style
    "Fixed Deposit premature closure attracts a penalty charge of 1 percent as per RBI guidelines.",    # 1: Policy style, same fact
    "Step 3 of the withdrawal SOP: apply a 1 percent penalty deduction on the FD interest rate.",         # 2: SOP style, same fact again
    "The premature withdrawal penalty is waived if the FD is closed due to the death of the depositor.", # 3: genuinely different (nominee exception)
    "Senior citizens receive an additional 0.5 percent interest rate on all Fixed Deposit tenures.",     # 4: genuinely different topic
]

QUERY = "what is the penalty for premature FD withdrawal"

def normalize_vector(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

# build dense vectors for query-relevance AND document-to-document
# similarity -- same embedding space, reused for both purposes
vectorizer = TfidfVectorizer()
sparse_matrix = vectorizer.fit_transform(CANDIDATE_POOL)
svd = TruncatedSVD(n_components=4, random_state=42)
doc_vectors = svd.fit_transform(sparse_matrix)
doc_vectors_norm = np.array([normalize_vector(v) for v in doc_vectors])

query_vector = normalize_vector(svd.transform(vectorizer.transform([QUERY]))[0])

# relevance of each candidate to the query -- this is the retrieval score
relevance_scores = np.array([cosine_similarity(query_vector, v) for v in doc_vectors_norm])

print("=" * 70)
print("CANDIDATE POOL AND RELEVANCE TO QUERY")
print("=" * 70)
print(f"Query: '{QUERY}'\n")
for i, (text, score) in enumerate(zip(CANDIDATE_POOL, relevance_scores)):
    print(f"  Doc {i} | relevance={score:.4f} | {text[:65]}...")

plain_top3 = np.argsort(relevance_scores)[::-1][:3]
print(f"\nPlain relevance-only top-3 (no diversity awareness): {list(plain_top3)}")
print("Notice: Docs 0, 1, 2 are very likely to be the top-3, since they")
print("all restate the SAME fact and score similarly high -- Doc 3's")
print("genuinely different, useful nominee-exception information gets")
print("excluded purely because it scores a bit lower in isolation.")
print("\nModule 1 complete. Run Module 2 next.")


CANDIDATE POOL AND RELEVANCE TO QUERY
Query: 'what is the penalty for premature FD withdrawal'

  Doc 0 | relevance=0.6506 | Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Doc 1 | relevance=0.2038 | Fixed Deposit premature closure attracts a penalty charge of 1 pe...
  Doc 2 | relevance=0.7180 | Step 3 of the withdrawal SOP: apply a 1 percent penalty deduction...
  Doc 3 | relevance=0.9343 | The premature withdrawal penalty is waived if the FD is closed du...
  Doc 4 | relevance=0.0012 | Senior citizens receive an additional 0.5 percent interest rate o...

Plain relevance-only top-3 (no diversity awareness): [np.int64(3), np.int64(2), np.int64(0)]
Notice: Docs 0, 1, 2 are very likely to be the top-3, since they
all restate the SAME fact and score similarly high -- Doc 3's
genuinely different, useful nominee-exception information gets
excluded purely because it scores a bit lower in isolation.

Module 1 complete. Run Module 2 next.


### Module 2: The MMR Algorithm

Built from scratch, applied to the candidate pool from Module 1.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: The MMR algorithm
#
# MMR score(d) = lambda * relevance(d, query)
#                - (1 - lambda) * max_similarity(d, already_selected)
# ------------------------------------------------------------------

def mmr_select(candidate_indices: list, relevance_scores: np.ndarray,
               doc_vectors: np.ndarray, k: int, lambda_weight: float = 0.6) -> list:
    """
    Greedily selects k documents from candidate_indices, balancing
    relevance to the query against redundancy with already-selected docs.
    """
    remaining = list(candidate_indices)
    selected = []

    while remaining and len(selected) < k:
        best_score = -float("inf")
        best_doc = None

        for doc_id in remaining:
            relevance_term = relevance_scores[doc_id]

            if selected:
                # redundancy penalty: highest similarity to anything already picked
                redundancy_term = max(
                    cosine_similarity(doc_vectors[doc_id], doc_vectors[s]) for s in selected
                )
            else:
                redundancy_term = 0.0  # nothing selected yet -- no penalty possible

            mmr_score = lambda_weight * relevance_term - (1 - lambda_weight) * redundancy_term

            if mmr_score > best_score:
                best_score = mmr_score
                best_doc = doc_id

        selected.append(best_doc)
        remaining.remove(best_doc)

    return selected


all_indices = list(range(len(CANDIDATE_POOL)))
mmr_result = mmr_select(all_indices, relevance_scores, doc_vectors_norm, k=3, lambda_weight=0.6)

print("=" * 70)
print("MMR SELECTION (lambda=0.6, k=3)")
print("=" * 70)
print(f"Query: '{QUERY}'\n")
for rank, doc_id in enumerate(mmr_result, start=1):
    print(f"  MMR pick {rank}: Doc {doc_id} | relevance={relevance_scores[doc_id]:.4f} | {CANDIDATE_POOL[doc_id][:60]}...")

print(f"\nPlain relevance-only top-3 was: {list(plain_top3)}")
print(f"MMR top-3 is:                   {mmr_result}")
if set(mmr_result) != set(plain_top3):
    print("\nMMR selected a DIFFERENT set -- it likely swapped out one of the")
    print("redundant penalty-restatement docs for the genuinely distinct")
    print("nominee-exception document (Doc 3), because that document's lower")
    print("relevance score was outweighed by its low redundancy with picks")
    print("already made.")
else:
    print("\nMMR selected the SAME set as plain relevance ranking this time --")
    print("this can happen if lambda is high enough that relevance still")
    print("dominates, or if the redundant docs are not similar enough to")
    print("each other to trigger a meaningful penalty. Try Module 3 to see")
    print("how changing lambda affects this.")
print("\nModule 2 complete. Run Module 3 next.")


MMR SELECTION (lambda=0.6, k=3)
Query: 'what is the penalty for premature FD withdrawal'

  MMR pick 1: Doc 3 | relevance=0.9343 | The premature withdrawal penalty is waived if the FD is clos...
  MMR pick 2: Doc 0 | relevance=0.6506 | Premature withdrawal of FD incurs a 1 percent penalty on the...
  MMR pick 3: Doc 2 | relevance=0.7180 | Step 3 of the withdrawal SOP: apply a 1 percent penalty dedu...

Plain relevance-only top-3 was: [np.int64(3), np.int64(2), np.int64(0)]
MMR top-3 is:                   [3, 0, 2]

MMR selected the SAME set as plain relevance ranking this time --
this can happen if lambda is high enough that relevance still
dominates, or if the redundant docs are not similar enough to
each other to trigger a meaningful penalty. Try Module 3 to see
how changing lambda affects this.

Module 2 complete. Run Module 3 next.


### Module 3: The Effect of Lambda

Runs MMR at several lambda values on the same candidate pool, showing the shift from pure relevance to pure diversity.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Effect of lambda on the final selection
# ------------------------------------------------------------------

print("=" * 70)
print("MMR RESULTS ACROSS DIFFERENT LAMBDA VALUES (k=3)")
print("=" * 70)
print(f"Query: '{QUERY}'\n")

for lambda_weight in [1.0, 0.7, 0.5, 0.3, 0.0]:
    result = mmr_select(all_indices, relevance_scores, doc_vectors_norm, k=3, lambda_weight=lambda_weight)
    labels = [f"Doc{d}" for d in result]
    print(f"  lambda={lambda_weight:.1f} -> {labels}")

print("\nAt lambda=1.0: pure relevance ranking, MMR degenerates to plain top-k.")
print("As lambda decreases: redundant documents get pushed out in favor of")
print("documents that are more different from what is already selected,")
print("even if those documents are individually less relevant to the query.")
print("At lambda=0.0: relevance is ignored entirely after the first pick --")
print("purely maximizing difference from prior picks, which can pull in")
print("something only weakly related to the query at all.")
print("\nThis is the concrete, measurable version of the theory's claim:")
print("there is no single 'correct' lambda -- it must be chosen based on")
print("how much the corpus actually needs a diversity correction, tuned")
print("by qualitatively reviewing real outputs like the ones printed above.")
print("\nModule 3 complete. All Topic 6 modules done.")

print()
print("=" * 70)
print("CHAPTER 7 TOPIC 6 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  MMR re-ranks an ALREADY-RETRIEVED candidate pool -- it does not
  retrieve new candidates.

  MMR_score(d) = lambda * relevance(d, query)
                 - (1 - lambda) * max_similarity(d, already_selected)

  The FIRST pick is always the most relevant document -- the redundancy
  term is zero until something has been selected.

  lambda=1 -> pure relevance (no diversity). lambda=0 -> pure diversity
  (ignores relevance after the first pick). Typical range: 0.5-0.7.

  Redundancy check MUST use document-to-document dense similarity,
  never a query-relevance score or keyword-based score.

  Only adopt MMR after confirming redundancy is a REAL, measured
  problem in actual retrieval outputs -- not by default.
""")


MMR RESULTS ACROSS DIFFERENT LAMBDA VALUES (k=3)
Query: 'what is the penalty for premature FD withdrawal'

  lambda=1.0 -> ['Doc3', 'Doc2', 'Doc0']
  lambda=0.7 -> ['Doc3', 'Doc2', 'Doc0']
  lambda=0.5 -> ['Doc3', 'Doc0', 'Doc1']
  lambda=0.3 -> ['Doc3', 'Doc1', 'Doc0']
  lambda=0.0 -> ['Doc0', 'Doc1', 'Doc4']

At lambda=1.0: pure relevance ranking, MMR degenerates to plain top-k.
As lambda decreases: redundant documents get pushed out in favor of
documents that are more different from what is already selected,
even if those documents are individually less relevant to the query.
At lambda=0.0: relevance is ignored entirely after the first pick --
purely maximizing difference from prior picks, which can pull in
something only weakly related to the query at all.

This is the concrete, measurable version of the theory's claim:
there is no single 'correct' lambda -- it must be chosen based on
how much the corpus actually needs a diversity correction, tuned
by qualitatively reviewing real o